# Learning Objectives

In this notebook, you will craft sophisticated ETL jobs that interface with a variety of common data sources, such as 
- REST APIs (HTTP endpoints)
- RDBMS
- Hive tables (managed tables)
- Various file formats (csv, json, parquet, etc.)

d

# Interview Questions

As you progress through the practice, attempt to answer the following questions:

## Columnar File
- What is a columnar file format and what advantages does it offer?
- Why is Parquet frequently used with Spark and how does it function?
- How do you read/write data from/to a Parquet file using a DataFrame?

## Partitions
- How do you save data to a file system by partitions? (Hint: Provide the code)
- How and why can partitions reduce query execution time? (Hint: Give an example)

## JDBC and RDBMS
- How do you load data from an RDBMS into Spark? (Hint: Discuss the steps and JDBC)

## REST API and HTTP Requests
- How can Spark be used to fetch data from a REST API? (Hint: Discuss making API requests)

## ETL Job One: Parquet file
### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Data transformation requirements https://pgexercises.com/questions/aggregates/fachoursbymonth.html

### Load
Load data into a parquet file

### What is Parquet? 

Columnar files are an important technique for optimizing Spark queries. Additionally, they are often tested in interviews.
- https://www.youtube.com/watch?v=KLFadWdomyI
- https://www.databricks.com/glossary/what-is-parquet

In [0]:
%sql
-- CREATE CATALOG IF NOT EXISTS training;
-- CREATE SCHEMA IF NOT EXISTS training.raw;
CREATE VOLUME IF NOT EXISTS training.raw.processed;

In [0]:
# Write your solution here (ETL Job 1)

from pyspark.sql.functions import col, sum

bookings_path = "/Volumes/training/raw/files/bookings.csv"

# ---------------------------------------------- Extract -----------------------------------------------------
bookings_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(bookings_path)

# ---------------------------------------------- Transform ---------------------------------------------------

# Only applying transformations on bookings DataFrame. Other DataFrames (members, facilities) stay the same.

transformed_bookings_df = bookings_df.filter((col("starttime") >= "2012-09-01") & (col("starttime") < "2012-10-01")).groupBy("facid").agg(sum("slots").alias("Total Slots")).orderBy("Total Slots", ascending=True)

# ------------------------------------------------ Load ------------------------------------------------------

# Writing (Loading) as parquet files

transformed_bookings_df.write.mode("overwrite").parquet("/Volumes/training/raw/processed/bookings.parquet")


In [0]:
# Display Parquet File

parquet_df = spark.read.parquet("/Volumes/training/raw/processed/bookings.parquet")
display(parquet_df)

facid,Total Slots
5,122
3,422
7,426
8,471
6,540
2,570
1,588
0,591
4,648


## ETL Job Two: Partitions

### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Transform the data https://pgexercises.com/questions/joins/threejoin.html

### Load
Partition the result data by facility column and then save to `threejoin_delta` managed table. Additionally, they are often tested in interviews.

hint: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html

What are paritions? 

Partitions are an important technique to optimize Spark queries
- https://www.youtube.com/watch?v=hvF7tY2-L3U&t=268s

In [0]:
# Write your solution here (ETL Job 2)

import tempfile
import os
from pyspark.sql.functions import col, concat, lit

bookings_path = "/Volumes/training/raw/files/bookings.csv"
facilities_path = "/Volumes/training/raw/files/facilities.csv"
members_path = "/Volumes/training/raw/files/members.csv"

# ---------------------------------------------- Extract -----------------------------------------------------
bookings_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(bookings_path)

facilities_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(facilities_path)

members_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(members_path)

# ---------------------------------------------- Transform ---------------------------------------------------

facilities_filtered = facilities_df.filter((facilities_df.name == "Tennis Court 1") | (facilities_df.name == "Tennis Court 2"))

member_names = concat(members_df.firstname, lit(" "), members_df.surname).alias("member")

facility = facilities_df.name.alias("facility")

transformed_df = bookings_df.join(members_df, bookings_df.memid == members_df.memid).join(facilities_filtered, bookings_df.facid == facilities_filtered.facid).select(member_names, facility, bookings_df.facid).distinct().orderBy(member_names, facility)

# ------------------------------------------------ Load ------------------------------------------------------

output_path = "/Volumes/training/raw/processed/transformed_df_partitioned"

transformed_df.write.partitionBy("facid").mode("overwrite").format("parquet").save(output_path)

parquet_df_t = spark.read.parquet(output_path)
display(parquet_df_t)



member,facility,facid
Tracy Smith,Tennis Court 1,0
Timothy Baker,Tennis Court 1,0
Tim Rownam,Tennis Court 1,0
Tim Boothe,Tennis Court 1,0
Ramnaresh Sarwin,Tennis Court 1,0
Ponder Stibbons,Tennis Court 1,0
Nancy Dare,Tennis Court 1,0
Matthew Genting,Tennis Court 1,0
John Hunt,Tennis Court 1,0
Joan Coplin,Tennis Court 1,0


## ETL Job Three: HTTP Requests

### Extract
Extract daily stock price data price from the following companies, Google, Apple, Microsoft, and Tesla. 

Data Source
- API: https://rapidapi.com/alphavantage/api/alpha-vantage
- Endpoint: GET `TIME_SERIES_DAILY`

Sample HTTP request

```
curl --request GET \
	--url 'https://alpha-vantage.p.rapidapi.com/query?function=TIME_SERIES_DAILY&symbol=TSLA&outputsize=compact&datatype=json' \
	--header 'X-RapidAPI-Host: alpha-vantage.p.rapidapi.com' \
	--header 'X-RapidAPI-Key: [YOUR_KEY]'

```

Sample Python HTTP request

```
import requests

url = "https://alpha-vantage.p.rapidapi.com/query"

querystring = {
    "function":"TIME_SERIES_DAILY",
    "symbol":"IBM",
    "datatype":"json",
    "outputsize":"compact"
}

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "[YOUR_KEY]"
}

response = requests.get(url, headers=headers, params=querystring)

data = response.json()

# Now 'data' contains the daily time series data for "IBM"
```

### Transform
Find **weekly** max closing price for each company.

hints: 
  - Use a `for-loop` to get stock data for each company
  - Use the spark `union` operation to concat all data into one DF
  - create a new `week` column from the data column
  - use `group by` to calcualte max closing price

### Load
- Partition `DF` by company
- Load the DF in to a managed table called, `max_closing_price_weekly`

In [0]:
# Write your solution here (ETL Job 3)

api_key = dbutils.secrets.get(scope="rapidapi_key", key="rapid_api_key")

import requests
import json
import os
import time
import pandas as pd
from pyspark.sql.functions import weekofyear, year, max as spark_max, col, lit

# ---------------------------------------------- Extract -----------------------------------------------------
url = "https://alpha-vantage.p.rapidapi.com/query"
companies = ["GOOG", "AAPL", "MSFT", "TSLA"]
all_data = []

for symbol in companies:
    querystring = {
                "function":"TIME_SERIES_DAILY",
                "symbol": symbol,
                "outputsize":"compact",
                "datatype":"json"
                }
    
    headers = {
	"x-rapidapi-key": api_key,
	"x-rapidapi-host": "alpha-vantage.p.rapidapi.com"
    }

    response = requests.get(url, headers=headers, params=querystring)
    
    data = response.json().get("Time Series (Daily)", {})

    if not data:
        print(f"No data for {symbol}, skipping")
        continue

    rows = []
    for date_str, values in data.items():
        row = {
            "date": date_str,
            "close": float(values["4. close"]),
            "company": symbol
        }
        rows.append(row)
    
    temp_df = pd.DataFrame(rows)
    all_data.append(temp_df)

    time.sleep(12)

pandas_df = pd.concat(all_data)

# ---------------------------------------------- Transform ---------------------------------------------------
df = spark.createDataFrame(pandas_df)

df = df.withColumn("week", weekofyear(col("date"))).withColumn("year", year(col("date")))

max_closing_price_weekly = df.groupBy("company", "year", "week").agg(spark_max("close").alias("max_closing_price"))

# ------------------------------------------------ Load ------------------------------------------------------
max_closing_price_weekly.write.mode("overwrite").saveAsTable("max_closing_price_weekly")

display(spark.table("max_closing_price_weekly"))



company,year,week,max_closing_price
GOOG,2025,34,206.72
GOOG,2025,38,251.76
GOOG,2025,36,235.17
GOOG,2025,32,202.09
GOOG,2025,30,194.08
GOOG,2025,29,185.94
GOOG,2025,35,213.53
GOOG,2025,37,241.38
GOOG,2025,33,204.91
GOOG,2025,31,197.44


## ETL Job Four: RDBMS


### Extract
Extract RNA data from a public PostgreSQL database.

- https://rnacentral.org/help/public-database
- Extract 100 RNA records from the `rna` table (hint: use `limit` in your sql)
- hint: use `spark.read.jdbc` https://docs.databricks.com/external-data/jdbc.html

### Transform
We want to load the data as it so there is no transformation required.


### Load
Load the DF in to a managed table called, `rna_100_records`

In [0]:
# Write your solution here (ETL Job 4)

# ---------------------------------------------- Extract -----------------------------------------------------
db_url = "jdbc:postgresql://hh-pgsql-public.ebi.ac.uk:5432/pfmegrnargs"
db_username = "reader"
db_password = "NWDMCE5xdipIjRrp"

rna_100_records = (spark.read.format("jdbc")
            .option("url", db_url)
            .option("dbtable", "(SELECT * FROM rna LIMIT 100) as rna_100")
            .option("user", db_username)
            .option("password", db_password)
            .option("driver", "org.postgresql.Driver")
            .load()
            )

# --------------------------------------------- Transform ----------------------------------------------------
# No transformation needed as per requirements, we are just loading the data as a managed table.

# ----------------------------------------------- Load -------------------------------------------------------
rna_100_records.write.mode("overwrite").saveAsTable("rna_100_records")

display(rna_100_records)

id,upi,timestamp,userstamp,crc64,len,seq_short,seq_long,md5
8976530,URS000088F892,2015-10-20T18:04:07.000Z,RNACEN,602DA4B6D99778A0,1395,CTTACACATGCAAGTCGAGGGGTAACGATGGCTTCGGCCATGCGACGACCGGCGCACGGGTGAGTATCGCGTTTGTAACCTACCTTTAACTGGAAGATAGCCCGAAGAAATTCGGATTAATACTCCATAACACTATAGTCTGGCATCATTTTATAGTTAAAGCTCCGGCGGTTAAAGATGGGCATGCGTAACATTAGCTTGTTGGTGAGGTAACGGCTCACCAAGGCTACGATGTTTAGGGGGTCTGAGAGGATGGTCCCCCACACTGGTACTGAGACACGGACCAGACTCCTACGGGAGGCAGCAGTGAGGAATATTGGTCAATGGGCGAAAGCCTGAACCAGCCATGCCGCGTGAAGGATGACGGCGCTATGCGTTGTAAACTTCTTTTATACGGGAAGAAACCGTCCTACGTGTAGGGCGTTGACGGTACCGTACGAATAAGCATCGGCTAACTCCGTGCCAGCAGCCGCGGTAATACGGAGGATGCAAGCGTTATCCGGATTTATTGGGTTTAAAGGGTGCGTAGGCGGAAAATTAAGTCAGTGGTGAAATCCTGCGGCTCAACCGTAGAACTGTCATTGATACTGATTTTCTTGAATACAGTTGAGGTAGGCGGAATGTGTAATGTAGCGGTGAAATGCTTAGATATTACACAGAACACCGATTGCGAAGGCAGCTTACTAAACTGTGATTGACGCTGAGGCACGAAAGCGTGGGGAGCGAACAGGATTAGATACCCTGGTAGTCCACGCCGTAAACGATGATCACTCGTTGTTGGCAATACATCGTCAGCGACTGAGCGAAAGCATTAAGTGATCCACCTGGGGAGTACGATCGCAAGATTGAAACTCAAAGGAATTGACGGGGGCCCGCACAAGCGGAGGAACATGTGGTTTAATTCGATGATACGCGAGGAACCTTACCTGGGCTTAAATGTGGAACGACGGACCGGGAAACTGGTCTTTCTTCGGACGTTCTGCAAGGTGCTGCATGGCTGTCGTCAGCTCGTGCCGTGAGGTGTCGGGTTAAGTCCCATAACGAGCGCAACCCTCATCATTAGTTGCCAGCGGGTAATGCCGGGGACTCTAATGAAACTGCCGGTGTAAACCGAGAGGAAGGTGGGGATGACGTCAAGTCAGCACGGCCCTTATGTCCAGGGCTACACACGTGTTACAATGGTCAGTACAAAGGGCAGCTACCATGCGAATGGATGCTAATCTCGAAAACTGATCTCAGTTCGGATCGAAGTCTGCAACTCGACTTCGTGAAGTTGGATTCGCTAGTAATCGCGCATCAGCCACGGCGCGGTGAATACGTTCCCGGGCCTTGTACACACCGCCCGTCAAGCCATGGAAGCCGGGGGTACCTGAAGTTCGTAACCGCAAGGAGCGA,null,98c80f8ce896c071e0e1f945b9492b44
8976531,URS000088F893,2015-10-20T18:04:07.000Z,RNACEN,E37E40E1B378A91C,1417,TGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAGCGGAGATAAGATAAAAGCTTGCTTTTAACTTATCTTAGCGGCGGACGGGTGAGTAACGTGTGGGCAACCTGCCTCATACAGAGGGATAATCATGTGAAAACGTGACTAATACCTCATGTCATTGCTAGAAGGCATCTTCAGGCAAGAAAAGGAGTAATCCGGTATGAGATGGGCCCGCATCTGATTAGCTAGTTGGTGAGATAATAGCCCACCAAGGCAACGATCAGTAGCCGACCTGGGAGGGTGATCGGCCACATTGGGACTGAGACACGGCCCAAACTCCTACGGGAGGCAGCAGTGGGGAATATTGCACAATGGGGGAAACCCTGATGCAGCAACGCCGCGTGAAGGAAGACGGTTTTCGGATTGTAAACTTCTATCAATAGGGACGAAAGAAATGACGGTACCTAAATAAGAAGCCCCGGCTAACTACGTGCCAGCAGCCGCGGTAATACGTAGGGGGCAAGCGTTATCCGGAATTACTGGGTGTAAAGGGTGAGTAGGCGGCATGGCAAGTAAGATGTGAAAGCCCGAGGCTTAACCTCGGGATTGCATTTTAAACTGCTAAGCTAGAGTACAGGAGAGGAAAGCGGAATTCCTAGTGTAGCGGTGAAATGCGTAGATATTAGGAAGAACACCAGTGGCGAAGGCGGCTTTCTGGACTGGAAACTGACGCTGAGGCACGAAAGCGTGGGGAGCGAACAGGATTAGATACCCTGGTAGTCCACGCCGTAAACGATGAGTGCTAGGTGTCGGAGGGGAACCTTCGGTGCCGCAGCAAACGCAATAAGCACTCCACTTGGGGAGTACGATCGCAAGATTGAAACTCAAAGGAATTGACGGGGGCCCGCACAAGCGGTGGAGCATGTGGTTTAATTAGAAGCAACGCGAAGAACCTTATCAAGGCTTGACATCCCGATGACGAATACAGAGATGTATTTTCTTTTCGGAGCATCGGTGACAGGTGGTGCATGGTTGTTGTCAGCTCGTGTCGTGAGATGTTGGGTTAAGTCCCGCAACGAGCGCAACCCTTATCTTTAGTAGCCAGCAGTAAGATGGGCACTCTAAAGAGTATTCCGTGGATAACACGGAGGAAGGTGGGGATGACGTCAAATCATCATGCCCCTTATGTTTTTGGCATCACACGTGCTACAATTGATGGAACAAAGTGAGGCAAGACCGCGAGGTTAAGCAAAGCACAAAAACCCGTTCTCAGTTCGGATTGTAGTATGCAACTCGTCCACATGAAGCTGGAATTGTTAGTAATCGGGAATCAGAATGTCGCGGTGAATACGTTCCCGGGCCTTGTTCACACCCCCCGTCACACCATGGGAGTTGGAAGCACCCGAAGCCTGTGACCCAACCTGCAAAGGAGGGAGC,null,9232a3f3af1f517a352d163642ee2b1b
8976532,URS000088F894,2015-10-20T18:04:07.000Z,RNACEN,9497B60215B6D77F,1405,TAGTGGCGGACGGGTGAGTAACGCGTGAGTAACCTGCCTTTCAGAGGGGGATAACGTTCTGAAAAGAACGCTAATACCGCATAACATCAATTTATCGCATGATAGGTTGATCAAAGGAGCAATCCGCTGGAAGATGGACTCGCGTCCGATTAGCCAGTTGGCGGGGTAACGGCCCACCAAAGCGACGATCGGTAGCCGGACTGAGAGGTTGAACGGCCACATTGGGACTGAGACACGGCCCAGACTCCTACGGGAGGCAGCAGTGGGGGATATTGCACAATGGGGGAAACCCTGATGCAGCAACGCCGCGTGAGGGAAGAAGGTTTTCGGATTGTAAACCTCTGTTCTTAGTGACGATAATGACGGTAGCTAAGGAGAAAGCTCCGGCTAACTACGTGCCAGCAGCCGCGGTAATACGTAGGGAGCGAGCGTTGTCCGGATTTACTGGGTGTAAAGGGTGCGTAGGCGGCGAGGCAAGTCAGGCGTGAAATCTATGGGCTTAACCCATAAACTGCGCTTGAAACTGTCTTGCTTGAGTGAAGTAGAGGTAGGCGGAATTCCCGGTGTAGCGGTGAAATGCGTAGAGATCGGGAGGAACACCAGTGGCGAAGGCGGCCTACTGGGCTTTAACTGACGCTGAAGCACGAAAGCATGGGTAGCAAACAGGATTAGATACCCTGGTAGTCCATGCCGTAAACGATGATTACTAGGTGTGGGGGGTCTGACCCCCTCCGTGCCGCAGTTAACACAATAAGTAATCCACCTGGGGAGTACGGCCGCAAGGTTGAAACTCAAAGGAATTGACGGGGGCCCGCACAAGCA